In [1]:
import requests
import pandas as pd
import time
import random

# ======================
# CONFIG
# ======================
CLIENT_ID = "05c9f985c6e952bb896b822d18d0364f"
headers = {"X-MAL-CLIENT-ID": CLIENT_ID}

ranking_url = "https://api.myanimelist.net/v2/anime/ranking"

TOTAL_ANIME = 1000
BATCH_SIZE = 100

anime_ids = []
rows = []

# Getting Anime IDs
for offset in range(0, TOTAL_ANIME, BATCH_SIZE):
    print(f"\nFetching ranking offset: {offset}")

    params = {
        "ranking_type": "all",
        "limit": BATCH_SIZE,
        "offset": offset
    }

    attempt = 0
    while True:
        try:
            response = requests.get(
                ranking_url,
                headers=headers,
                params=params,
                timeout=15
            )
            response.raise_for_status()
            data = response.json()

            # Validate response
            if "data" not in data or not data["data"]:
                raise ValueError("Invalid ranking response")

            break

        except (requests.exceptions.RequestException, ValueError) as e:
            attempt += 1
            print(f"Ranking error (attempt {attempt}): {e}")
            sleep_time = min(20, 2 + attempt * 2 + random.random())
            print(f"Retrying in {sleep_time:.1f}s...")
            time.sleep(sleep_time)

    for item in data["data"]:
        anime_ids.append(item["node"]["id"])

    print(f"Collected so far: {len(anime_ids)}")
    time.sleep(1.0)

print(f"\nCollected {len(anime_ids)} anime IDs")

#Fetching data
for i, anime_id in enumerate(anime_ids, start=1):
    print(f"\n{i}/{len(anime_ids)} -> Fetching ID: {anime_id}")

    detail_url = f"https://api.myanimelist.net/v2/anime/{anime_id}"
    params = {
        "fields": "id,title,mean,rank,popularity,num_episodes,start_date,end_date,synopsis,media_type,genres,studios,source,start_season"
    }

    attempt = 0
    while True:
        try:
            response = requests.get(
                detail_url,
                headers=headers,
                params=params,
                timeout=30
            )
            response.raise_for_status()
            node = response.json()

            # Validate response
            if not node or "id" not in node or "title" not in node:
                raise ValueError("Invalid detail response")

            print(f"✅ Success for ID {anime_id}")
            break

        except (requests.exceptions.RequestException, ValueError) as e:
            attempt += 1
            print(f"Error (attempt {attempt}): {e}")

            sleep_time = min(20, 2 + attempt * 2 + random.random())
            print(f"Retrying in {sleep_time:.1f}s...")
            time.sleep(sleep_time)

    start_date = node.get("start_date")
    year = start_date[:4] if start_date else None

    rows.append({
        "id": node.get("id"),
        "title": node.get("title"),
        "episodes": node.get("num_episodes"),
        "rating": node.get("mean"),
        "rank": node.get("rank"),
        "popularity": node.get("popularity"),
        "start_date": start_date,
        "end_date": node.get("end_date"),
        "year": year,
        "synopsis": node.get("synopsis"),
        "media_type": node.get("media_type"),
        "genres_raw": str(node.get("genres", [])),
        "studios_raw": str(node.get("studios", [])),
        "source": node.get("source")
    })

    time.sleep(0.5)

df = pd.DataFrame(rows)
df.to_csv("top_1000_anime.csv", index=False)

print("\nSaved: top_1000_anime.csv")
print(df.head())


Fetching ranking offset: 0
Collected so far: 100

Fetching ranking offset: 100
Collected so far: 200

Fetching ranking offset: 200
Collected so far: 300

Fetching ranking offset: 300
Collected so far: 400

Fetching ranking offset: 400
Collected so far: 500


KeyboardInterrupt: 